# 第 9 课：科研代码不止能运行，还要可复现、可验证

**学习目标**：熟悉实验配置、随机种子、记录与最小单元测试。这是从“写 notebook”到“构建研究软件”的关键过渡。

当你比较 PPO 与 TD3/SAC 或代理模型方法时，需要记录：环境版本、CFD 物理设置、网格与时间步、训练数据来源、奖励定义、seed、策略 checkpoint、最终 CFD 测试结果。

不要只保存最好的曲线图片。保存原始数据、代码版本、配置和评估脚本。

In [ ]:
import numpy as np
from dataclasses import dataclass, asdict
import json, tempfile, pathlib

@dataclass(frozen=True)
class ExperimentConfig:
    seed: int = 2026
    reynolds_number: float = 100.0
    cfd_dt: float = 0.005
    control_stride: int = 20
    gamma: float = 0.99
    ppo_clip: float = 0.2

cfg=ExperimentConfig()
assert cfg.cfd_dt > 0 and cfg.control_stride > 0
print('Control physical time:',cfg.cfd_dt*cfg.control_stride)
print(json.dumps(asdict(cfg),ensure_ascii=False,indent=2))

## 9.1 利用不变量阻止“好看但错误”的结果

当数值求解器或 RL 实现出错时，奖励曲线依然可能向上，因此需要更接近数学定义的自动化测试。

In [ ]:
def discounted_return(rewards, gamma):
    future=0.0
    for reward in reversed(rewards):
        future = float(reward) + gamma * future
    return future

assert np.isclose(discounted_return([1.,2.,3.],.5),1+.5*2+.25*3)
assert np.isclose(discounted_return([5.],.99),5.)

def verify_transition(obs, action, reward, next_obs):
    assert obs.ndim == 1 and next_obs.shape == obs.shape
    assert np.all(np.isfinite(obs)) and np.all(np.isfinite(next_obs))
    assert np.all(np.isfinite(action)) and np.isfinite(reward)

verify_transition(np.zeros(8),np.array([0.]),0.,np.ones(8))
print('Invariant tests passed')

## 9.2 一次实验不能说明性能

要跨随机种子报告统计量，并将训练过程监控与最终评估分离。下例展示如何记录均值与不确定区间，但这里的数值是**合成测试数据，不是任何真实 PPO/CFD 结果**。

In [ ]:
rng=np.random.default_rng(22)
seeds=list(range(10))
synthetic_drag_reduction=rng.normal(loc=.085,scale=.025,size=len(seeds))
mean=float(synthetic_drag_reduction.mean())
sem=float(synthetic_drag_reduction.std(ddof=1)/np.sqrt(len(seeds)))
print(f'Synthetic-only mean reduction: {mean:.3f}; SEM: {sem:.3f}; n={len(seeds)}')

## 9.3 代码工程知识路线

Python 类型注解与 `dataclasses` → 包结构、模块导入与配置管理 → `pytest` → Git 分支/commit → `uv` 虚拟环境与锁文件 → 日志与 checkpoint → profiling → GPU、WSL2、Docker、HPC。

**优先学习顺序**：先会把 Notebook 实验整理到 `src/` 模块与 `tests/` 目录，再考虑复杂的多 GPU 并行。对于 CFD，单步守恒误差和收敛实验应该有独立的测试脚本。

### 练习

1. 将第 5 课的 PPO 训练过程拆分为 `env.py`、`policy.py`、`rollout.py`、`train.py` 和 `evaluate.py`。
2. 为你的代理模型设计回归测试：在一组固定轨迹上，新版本的阻力预测误差不能无解释地恶化。
3. 为每个策略保存 JSON config + 权重 + 多随机种子统计，并能完全复现。